# MODELO ARIMA SELIC

In [160]:
# teste realizado antes de definir o d do ARIMA.
def testar_estacionariedade(serie):
    """
    Aplica o teste de Dickey-Fuller Aumentado (ADF)
    para verificar presença de raiz unitária.
    
    H0: A série possui raiz unitária (não estacionária)
    H1: A série é estacionária
    """
    resultado = adfuller(serie.dropna())
    estatistica, p_valor = resultado[0], resultado[1]
    
    print("Estatística ADF:", estatistica)
    print("p-valor:", p_valor)
    
    if p_valor < 0.05:
        print("Conclusão: Série estacionária")
        return True
    else:
        print("Conclusão: Série não estacionária")
        return False


In [181]:
def ajustar_arima_diferenciado(serie):
    """
    Ajusta modelo ARIMA(1,1,0).
    
    Justificativa:
    Como a série apresentou raiz unitária,
    modelamos a primeira diferença.
    """
    modelo = SARIMAX(
        serie,
        order=(1,1,0),      # (p,d,q)
        trend='n',          # sem constante
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    resultado = modelo.fit(
        method='powell',   # método alternativo
        maxiter=5000,
        disp=False
    )
    
    print(resultado.summary())
    
    return resultado

In [182]:
def projetar_arima(modelo, serie, ano_final=2030):
    """
    Realiza projeção da taxa SELIC até o ano_final.
    
    A projeção é baseada exclusivamente
    na dinâmica estatística estimada.
    """
    ultimo_ano = serie.index[-1].year
    passos = (ano_final - ultimo_ano) * 12
    
    previsao = modelo.get_forecast(steps=passos)
    media_prevista = previsao.predicted_mean
    intervalo = previsao.conf_int()
    
    df_previsao = media_prevista.to_frame(name='selic_prevista')
    df_previsao['limite_inferior'] = intervalo.iloc[:,0]
    df_previsao['limite_superior'] = intervalo.iloc[:,1]
    
    return df_previsao

In [183]:
def plotar_projecao(df_mensal, df_previsao):
    """
    Plota série histórica mensal
    e projeção até 2030 com intervalo de confiança.
    """

    # Garantir que os índices são datetime
    df_mensal.index = pd.to_datetime(df_mensal.index)
    df_previsao.index = pd.to_datetime(df_previsao.index)

    # Garantir que as colunas são float
    df_mensal['valor'] = df_mensal['valor'].astype(float)
    df_previsao['selic_prevista'] = df_previsao['selic_prevista'].astype(float)
    df_previsao['limite_inferior'] = df_previsao['limite_inferior'].astype(float)
    df_previsao['limite_superior'] = df_previsao['limite_superior'].astype(float)

    plt.figure(figsize=(12,6))

    # Histórico
    plt.plot(
        df_mensal.index,
        df_mensal['valor'],
        label='Histórico Mensal',
        linewidth=2
    )

    # Projeção
    plt.plot(
        df_previsao.index,
        df_previsao['selic_prevista'],
        label='Projeção ARIMA(1,1,0)',
        linestyle='--',
        linewidth=2
    )

    # Intervalo de confiança
    plt.fill_between(
        df_previsao.index,
        df_previsao['limite_inferior'],
        df_previsao['limite_superior'],
        alpha=0.3,
        label='Intervalo de Confiança'
    )

    plt.axvline(df_mensal.index[-1], color='black', linestyle=':', label='Início da Projeção')

    plt.title('Projeção da Taxa SELIC até 2030 - ARIMA(1,1,0)')
    plt.xlabel('Ano')
    plt.ylabel('Taxa (%)')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
